# OSEMN Model — Students Performance Dataset

## Tapşırıq
Verilən mənbədən 1000 sətirlik data əldə etmək, həmin datanı emal etmək və OSEMN modelini tətbiq etmək.

Dataset: **Students Performance in Exams** — 1000 tələbə və 8 sütun.


## O — Obtain

Data CSV formatında açıq mənbədən əldə edilir.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/AyeshaNehal/Student-Performance-Analysis-EDA-Project-/main/StudentsPerformance.csv"
df = pd.read_csv(url)

print("Sətir sayı:", df.shape[0])
print("Sütun sayı:", df.shape[1])
print(df.head())


## S — Scrub

Boş qiymətlər, duplicate sətirlər və sütun tipləri yoxlanılır.

In [ ]:
print("Boş qiymətlər:")
print(df.isnull().sum())

print("Duplicate sətirlər:", df.duplicated().sum())
df = df.drop_duplicates().copy()

numeric_columns = ["math score", "reading score", "writing score"]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

print("\nTəmizlənmiş data:")
print(df.isnull().sum())


## E — Explore

Əsas statistik göstəricilər və dəyişənlər arasındakı əlaqələr araşdırılır.

In [ ]:
print(df.describe())

print("\nGender üzrə orta nəticələr:")
print(df.groupby("gender")[numeric_columns].mean())

print("\nTest preparation course üzrə orta nəticələr:")
print(df.groupby("test preparation course")[numeric_columns].mean())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.heatmap(df[numeric_columns].corr(), annot=True)
plt.title("Score Correlation")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df[numeric_columns])
plt.title("Score Distribution")
plt.show()


## M — Model

Linear Regression modeli ilə math score proqnozlaşdırılır. Kateqorik dəyişənlər One-Hot Encoding ilə rəqəmsallaşdırılır.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

target = "math score"
features = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course", "reading score", "writing score"]

X = df[features]
y = df[target]

categorical = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course"]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical)
], remainder="passthrough")

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


## N — iNterpret

MAE və RMSE proqnozların real math score-dan orta uzaqlığını göstərir. R² isə modelin math score dəyişkənliyinin nə qədər hissəsini izah etdiyini göstərir. Yuxarıdakı nəticələr əsasında modelin performansı şərh edilir.